# Blindless Detection · T4 · Tek defter

Tüm eğitim kodu aşağıdaki hücrelerde açıkça yer alır. Hücreleri **yukarıdan aşağıya** çalıştırın.
Çalışma zamanı ayarından **T4 GPU** seçin. Kod paketi veya veri ZIP'i yükleme adımı yoktur.

Veri klasörü: `/content/drive/MyDrive/aptos2019-blindness-detection`.
Başlangıç: **EfficientNet-B3 / ImageNet / 300 piksel**. B0 ve V2-S de ayarlardan seçilebilir.
Bu defteri açmak hücreleri tek tek kopyalama işini ortadan kaldırır; kodları isterseniz yeni bir deftere de kopyalayabilirsiniz.

Mevcut 2.452 / 526 / 526 ayrımı aynı algoritmayla yeniden oluşturulur ve kayıtlı özetiyle doğrulanır.
Bu bir araştırma prototipidir; model skorları klinik doğruluk olasılığı değildir.

## 1 · Ortam ve kütüphaneler
Bu hücre T4/CUDA erişimini kontrol eder.

In [ ]:
import sys, os, subprocess, importlib.util
from pathlib import Path
for module, requirement in [("sklearn", "scikit-learn>=1.5,<2"), ("PIL", "Pillow>=10"), ("matplotlib", "matplotlib>=3.8")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", requirement], check=True)
try:
    import torch, torchvision
except Exception as exc:
    raise RuntimeError("Colab PyTorch kurulumu uyumsuz. Temiz bir T4 çalışma zamanı açıp varsayılan PyTorch/Torchvision kurulumu ile yeniden dene.") from exc
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 5), "PyTorch >=2.5 gerekli"
assert torch.cuda.is_available(), "T4 GPU seçili değil. Çalışma zamanı ayarından GPU seç."
print("Python:", sys.version.split()[0], "Torch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0), "Bellek:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
import csv, hashlib, json, math, random, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from types import SimpleNamespace
from copy import copy
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, cohen_kappa_score,
                             confusion_matrix, f1_score)

## 2 · Drive bağlantısı ve veri klasörü
Drive erişimini onaylayın. Kaynak klasörde `train.csv` ve `train_images/` bulunmalı.
Eğitim, doğrulama ve son değerlendirme, etiketli görüntülerden ayrılan sabit kümeleri kullanır.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import csv

SOURCE = Path("/content/drive/MyDrive/aptos2019-blindness-detection")
if not SOURCE.is_dir():
    raise FileNotFoundError(
        f"Veri klasörü bulunamadı: {SOURCE}. Önce bu hücrede Drive bağlantısını açın."
    )
train_csv = SOURCE / "train.csv"
train_images = SOURCE / "train_images"
if not train_csv.is_file() or not train_images.is_dir():
    raise FileNotFoundError(
        f"{SOURCE} klasörü doğrudan train.csv dosyasını ve train_images klasörünü içermeli."
    )
with train_csv.open(encoding="utf-8-sig", newline="") as stream:
    reader = csv.DictReader(stream)
    if not {"id_code", "diagnosis"}.issubset(reader.fieldnames or []):
        raise ValueError("train.csv içinde id_code ve diagnosis sütunları bulunmalı.")
    labeled_count = sum(1 for _ in reader)
if labeled_count == 0:
    raise ValueError("train.csv boş; etiketli APTOS verisini kontrol edin.")
print("Veri kaynağı:", SOURCE)
print("Etiketli görüntü kaydı:", labeled_count)
print("Sonraki hazırlık adımı görüntüleri ve etiketleri doğrulayarak yerel önbelleği oluşturacak.")

## 3 · Model ve eğitim ayarları
İlk deneme B3/300, batch=12 ile başlar. V2-S için batch=8 seçilir. Bellek hatasında 8 veya 4 deneyin.
Aynı denemeye devam ederken ayarları değiştirmeyin; yeni ayarlar için yeni `RUN_NAME` kullanın.

In [ ]:
ARCHITECTURE = "efficientnet_b3"  # efficientnet_b0 / efficientnet_b3 / efficientnet_v2_s
RUN_NAME = "b3_300_seed42_v1"
SIZES = {"efficientnet_b0": 224, "efficientnet_b3": 300, "efficientnet_v2_s": 384}
BATCH_SIZE = 8 if ARCHITECTURE == "efficientnet_v2_s" else 12
EPOCHS = 30
MIXUP = 0.1
THRESHOLD = 0.80

if ARCHITECTURE not in SIZES:
    raise ValueError("Listelenen üç modelden birini seçin.")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in (".", ".."):
    raise ValueError("RUN_NAME tek klasör adı olmalı.")
if BATCH_SIZE < 1 or EPOCHS < 1 or MIXUP < 0 or not 0 < THRESHOLD < 1:
    raise ValueError("Batch, epoch, MixUp ve eşik ayarlarını kontrol edin.")

ROOT = Path("/content/BlindlessDetection")
RUN = Path("/content/drive/MyDrive/BlindlessDetection/runs") / RUN_NAME
CACHE = Path("/content/aptos_cache512")
ROOT.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
args = SimpleNamespace(
    source=SOURCE, manifests=ROOT/"data", cache=CACHE, cache_size=512,
    architecture=ARCHITECTURE, image_size=SIZES[ARCHITECTURE], output=RUN,
    batch_size=BATCH_SIZE, epochs=EPOCHS, workers=2, warmup_epochs=2,
    patience=6, seed=42, mixup=MIXUP, lr=1e-4, head_lr=5e-4,
    weight_decay=1e-4, threshold=THRESHOLD, action="train", resume=False,
    confirm_final_model=False,
)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN / "environment.txt").write_text(versions, encoding="utf-8")
print("Model:", ARCHITECTURE, "Girdi:", args.image_size, "Deneme:", RUN)

## 4 · Görüntü ön işleme ve model mimarileri
Retina kenarları kırpılır, kareye tamamlanır ve aynı normalleştirme masaüstü uygulamasında da kullanılır.
Bu hücre fonksiyonları tanımlar; ImageNet ağırlıkları eğitim başlarken indirilir.

In [ ]:
CLASSES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']


PREPROCESS = 'retina-border-crop-rgb-imagenet-v1'


MODEL_REGISTRY = {
    'efficientnet_b0': {'display_name': 'EfficientNet-B0', 'image_size': 224},
    'efficientnet_b3': {'display_name': 'EfficientNet-B3', 'image_size': 300},
    'efficientnet_v2_s': {'display_name': 'EfficientNetV2-S', 'image_size': 384},
}

def crop_retina(image):
    image = image.convert('RGB')
    a = np.asarray(image)
    mask = a.max(axis=2) > 12
    ys, xs = np.where(mask)
    if len(xs) and xs.max()-xs.min() > image.width//3 and ys.max()-ys.min() > image.height//3:
        image = image.crop((int(xs.min()), int(ys.min()), int(xs.max()+1), int(ys.max()+1)))
    # Pad rather than stretch the retina.
    side = max(image.size)
    square = Image.new('RGB', (side, side))
    square.paste(image, ((side-image.width)//2, (side-image.height)//2))
    return square

def transform(size=224, training=False):
    ops = [transforms.Resize((size, size))]
    if training:
        ops += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(20),
                transforms.ColorJitter(brightness=.12, contrast=.12)]
    return transforms.Compose(ops + [transforms.ToTensor(),
        transforms.Normalize([.485,.456,.406], [.229,.224,.225])])


def build_model(pretrained=False, architecture='efficientnet_b0'):
    """Build a supported 5-class model; inference never downloads weights."""
    if not isinstance(architecture, str) or architecture not in MODEL_REGISTRY:
        raise ValueError(f'Desteklenmeyen model mimarisi: {architecture}')
    if pretrained:
        torch.hub.set_dir(str(ROOT / '.cache' / 'torch'))
    constructors = {
        'efficientnet_b0': (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
        'efficientnet_b3': (models.efficientnet_b3, models.EfficientNet_B3_Weights.IMAGENET1K_V1),
        'efficientnet_v2_s': (models.efficientnet_v2_s, models.EfficientNet_V2_S_Weights.IMAGENET1K_V1),
    }
    constructor, weights = constructors[architecture]
    model = constructor(weights=weights if pretrained else None)
    model.classifier = nn.Sequential(nn.Dropout(.3), nn.Linear(model.classifier[1].in_features, 5))
    return model

## 5 · Veri hazırlama ve artırma fonksiyonları
Kod, birebir görüntü kopyalarını ayıklar ve aynı görüntüde çelişen etiketleri dışlar.
Artırma yalnızca eğitimde uygulanır. Bu hücre tanımları yükler; hazırlık bir sonraki hücrede başlar.

In [ ]:
def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temp, path)


def atomic_save(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temp)
    os.replace(temp, path)


def manifests(directory):
    """Ignore local paths; retain audited IDs, labels, original pixel hashes/order."""
    result, ids, hashes = {}, set(), set()
    for split in ("train", "val", "test"):
        with (directory / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
            rows = list(csv.DictReader(f))
        if not rows:
            raise ValueError(f"Empty split: {split}")
        for row in rows:
            ident, digest, label = row["id_code"], row["sha256"], int(row["diagnosis"])
            if not ident or any(c not in "0123456789abcdef" for c in ident.lower()):
                raise ValueError("Unexpected APTOS image ID")
            if label not in range(5) or len(digest) != 64:
                raise ValueError("Invalid label/hash")
            if ident in ids or digest in hashes:
                raise ValueError("Duplicate image ID/hash within or across audited splits")
            ids.add(ident)
            hashes.add(digest)
        result[split] = rows
    serial = {s: [{k: r[k] for k in ("id_code", "diagnosis", "sha256")} for r in rows]
              for s, rows in result.items()}
    digest = hashlib.sha256(json.dumps(serial, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    return result, digest

# Hücre: Orijinal APTOS verisini doğrula, sabit veri ayrımını yeniden oluştur.
# Görüntüler Google Drive'dan okunur; eğitim önbelleği yerel /content altında kalır.
from collections import Counter
from tempfile import TemporaryDirectory
from sklearn.model_selection import train_test_split


def recreate_audited_splits(records):
    """Eski denetimle aynı sıra, birebir kopya/çelişki kuralı ve seed=42."""
    seen = {}
    unique = []
    duplicates = 0
    conflicts = set()
    for record in records:
        digest = record["sha256"]
        if digest in seen:
            duplicates += 1
            if seen[digest]["diagnosis"] != record["diagnosis"]:
                conflicts.add(digest)
        else:
            seen[digest] = record
            unique.append(record)
    # Aynı piksellere farklı etiket verilmişse bu hash'in bütün örnekleri dışlanır.
    unique = [record for record in unique if record["sha256"] not in conflicts]
    train, rest = train_test_split(
        unique, test_size=.3,
        stratify=[record["diagnosis"] for record in unique], random_state=42,
    )
    validation, test = train_test_split(
        rest, test_size=.5,
        stratify=[record["diagnosis"] for record in rest], random_state=42,
    )
    parts = {"train": train, "val": validation, "test": test}
    audit = {
        "source_rows": len(records),
        "exact_duplicates_removed": duplicates,
        "conflicting_hashes": len(conflicts),
        "seed": 42,
        "limitations": "No patient identifiers: patient-level separation and near-duplicate exclusion are not guaranteed.",
    }
    for name, part in parts.items():
        audit[name] = {
            "count": len(part),
            "classes": dict(Counter(record["diagnosis"] for record in part)),
        }
    return parts, audit


def prepare_from_drive(args, expected_split_digest):
    """Orijinalleri okur; yalnızca önceki sabit ayrım birebir sağlanırsa hazır olur.

    args.source: Drive'daki train.csv ve train_images içeren klasör.
    args.manifests: Yerel train/val/test CSV çıktılarının klasörü.
    args.cache: Yerel görüntü önbelleğinin klasörü.
    args.cache_size: Yeni modeller için 512.
    Gerekli önceki hücreler: crop_retina, PREPROCESS, atomic_json, manifests.
    """
    if (not isinstance(expected_split_digest, str) or len(expected_split_digest) != 64
            or any(character not in "0123456789abcdef" for character in expected_split_digest)):
        raise ValueError("Önceki sabit ayrımın SHA-256 değeri gerekli; otomatik yeni ayrım seçilmez.")
    source = Path(args.source).resolve()
    manifest_dir = Path(args.manifests).resolve()
    cache = Path(args.cache).resolve()
    if not (source / "train.csv").is_file() or not (source / "train_images").is_dir():
        raise FileNotFoundError("Kaynak klasörde train.csv ve train_images bulunmalı: " + str(source))
    if source == manifest_dir or source in manifest_dir.parents or source == cache or source in cache.parents:
        raise ValueError("Önbellek ve manifest çıktıları orijinal veri klasörünün dışında olmalı.")
    if args.cache_size != 512:
        raise ValueError("Bu yeni eğitim akışı orijinal görüntülerden 512 piksel önbellek oluşturur.")
    with (source / "train.csv").open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if not {"id_code", "diagnosis"}.issubset(reader.fieldnames or []):
            raise ValueError("train.csv id_code ve diagnosis sütunlarını içermeli.")
        source_rows = list(reader)
    if not source_rows:
        raise ValueError("train.csv boş.")
    cache.mkdir(parents=True, exist_ok=True)
    manifest_dir.mkdir(parents=True, exist_ok=True)
    # Kesinti veya hash uyuşmazlığı eski önbelleği yanlışlıkla hazır bırakmasın.
    atomic_json({"status": "building", "cache_size": 512,
                 "expected_split_digest": expected_split_digest}, cache / "cache.json")
    records = []
    seen_ids = set()
    for index, row in enumerate(source_rows, start=1):
        ident = row["id_code"]
        if not ident or any(character not in "0123456789abcdef" for character in ident.lower()):
            raise ValueError("Geçersiz APTOS görüntü kimliği: " + str(ident))
        if ident in seen_ids:
            raise ValueError("train.csv içinde yinelenen id_code: " + ident)
        seen_ids.add(ident)
        label = int(row["diagnosis"])
        if label not in range(5):
            raise ValueError("Etiket 0–4 aralığında olmalı: " + ident)
        with Image.open(source / "train_images" / f"{ident}.png") as image:
            image.load()
            rgb = image.convert("RGB")
            # Eski prepare_data.py ile aynı hash: boyut + RGB piksel baytları.
            digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            target = cache / f"{ident}.png"
            temporary = cache / f"{ident}.tmp.png"
            crop_retina(rgb).resize((512, 512), Image.Resampling.LANCZOS).save(temporary)
            os.replace(temporary, target)
        records.append({"id_code": ident, "diagnosis": label, "sha256": digest, "path": str(target)})
        if index % 200 == 0:
            print(f"Orijinal görüntü denetimi / yerel önbellek: {index}/{len(source_rows)}", flush=True)
    parts, audit = recreate_audited_splits(records)
    # Önce geçici CSV'leri aynı manifests() işleviyle kontrol ederiz. Yerel dosya
    # yolları özete dahil değildir; kimlik, etiket, hash ve sıra birebir aynıdır.
    with TemporaryDirectory(prefix="split-check-", dir=manifest_dir) as temporary_directory:
        staging = Path(temporary_directory)
        for split, part in parts.items():
            with (staging / f"{split}.csv").open("w", encoding="utf-8", newline="") as stream:
                writer = csv.DictWriter(stream, fieldnames=["id_code", "diagnosis", "sha256", "path"])
                writer.writeheader()
                writer.writerows(part)
        verified_rows, actual_digest = manifests(staging)
        if actual_digest != expected_split_digest:
            raise ValueError(
                "Veri ayrımı eski sabit ayrımla uyuşmadı; eğitim başlatılmadı. "
                "Orijinal CSV sırasını, etiketleri, görüntüleri ve sklearn sürümünü kontrol et. "
                f"Beklenen: {expected_split_digest}; bulunan: {actual_digest}"
            )
        for split in ("train", "val", "test"):
            os.replace(staging / f"{split}.csv", manifest_dir / f"{split}.csv")
    audit.update(source=str(source), preprocess=PREPROCESS, split_digest=actual_digest)
    atomic_json(audit, manifest_dir / "audit.json")
    atomic_json({"status": "ready", "split_digest": actual_digest, "cache_size": 512,
                 "preprocess": PREPROCESS, "images": sum(len(rows) for rows in verified_rows.values())}, cache / "cache.json")
    print("HAZIR: önceki veri ayrımı birebir doğrulandı:",
          {split: len(rows) for split, rows in verified_rows.items()}, flush=True)
    return audit


def training_transform(size):
    # Mild scale/rotation changes keep the retina; no augmentation on validation/test.
    return transforms.Compose([
        transforms.Resize((size, size)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomAffine(degrees=20, scale=(.95, 1.05), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ColorJitter(brightness=.12, contrast=.12, saturation=.05), transforms.ToTensor(),
        transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
    ])


class Retinas(Dataset):
    def __init__(self, rows, cache, size, training=False):
        self.rows, self.cache = rows, cache
        self.transform = training_transform(size) if training else transform(size, False)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(self.cache / f'{row["id_code"]}.png') as image:
            x = self.transform(image.convert("RGB"))
        return x, int(row["diagnosis"])


def loader(rows, args, training=False, smoke=False):
    dataset = Retinas(rows, args.cache, args.image_size, training)
    if smoke:
        dataset = Subset(dataset, list(range(min(len(dataset), args.batch_size * 2))))
    # Fresh workers each epoch derive their seeds from the saved parent Torch RNG.
    return DataLoader(dataset, batch_size=args.batch_size, shuffle=training, num_workers=args.workers,
                      pin_memory=torch.cuda.is_available(), persistent_workers=False)


def checked_data(args):
    rows, digest = manifests(args.manifests)
    meta = json.loads((args.cache / "cache.json").read_text(encoding="utf-8"))
    if (meta.get("status") != "ready" or meta.get("split_digest") != digest or
            meta.get("preprocess") != PREPROCESS or meta.get("cache_size") != args.cache_size):
        raise ValueError("Cache incomplete or split/preprocessing mismatch. Run prepare.")
    return rows, digest

## 6 · Drive verisini hazırla
Drive'daki orijinaller bir kez okunur; 512 piksel önbellek Colab'ın yerel diskine yazılır.
İlk hazırlık zaman alabilir; görüntü sayacı ilerlemeyi gösterir. Kaynak görüntüler değiştirilmez.
Aynı veri ayrımı üretilemezse eğitim başlamadan açıklamalı hata verir. Yeni oturumda bu adımı yeniden çalıştırın.

In [ ]:
EXPECTED_SPLIT_DIGEST = "0be01ec9111f71888241979d4172392870b49391a2b684b8344ecfc970945511"
prepare_from_drive(args, EXPECTED_SPLIT_DIGEST)

## 7 · Ölçümler ve eğitim yardımcıları
Sınıf bazında duyarlılık, macro F1 ve QWK hesaplanır. Devam kaydına rastgelelik ve optimizer durumu da yazılır.

In [ ]:
def metrics(labels, logits):
    predictions = logits.argmax(1)
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic", labels=list(range(5)))
    return {"accuracy": float(accuracy_score(labels, predictions)),
            "macro_f1": float(f1_score(labels, predictions, labels=list(range(5)), average="macro", zero_division=0)),
            "qwk": float(qwk) if np.isfinite(qwk) else None,
            "confusion_matrix": confusion_matrix(labels, predictions, labels=list(range(5))).tolist(),
            "classes": classification_report(labels, predictions, labels=list(range(5)), target_names=CLASSES,
                                             output_dict=True, zero_division=0)}


def selective_metrics(labels, logits, temperature, threshold):
    scores, predicted = (logits / temperature).softmax(1).max(1)
    accepted = scores >= threshold
    return {"coverage": float(accepted.float().mean()), "accepted_count": int(accepted.sum()),
            "accepted_accuracy": float((predicted[accepted] == labels[accepted]).float().mean()) if accepted.any() else None,
            "temperature": temperature, "threshold": threshold,
            "note": "Softmax/coverage are research diagnostics, not medical reliability guarantees."}


def evaluate(model, data_loader, device):
    model.eval()
    labels, logits = [], []
    with torch.inference_mode():
        for x, y in data_loader:
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                output = model(x.to(device))
            logits.append(output.float().cpu())
            labels.append(y)
    return torch.cat(labels), torch.cat(logits)


def device_for(args):
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
        return torch.device("cuda")
    if args.action != "smoke":
        raise RuntimeError("Select T4 GPU in Colab. This notebook requires CUDA; TPU/XLA is not supported.")
    return torch.device("cpu")


def rng_state():
    state = np.random.get_state()
    return {"python": random.getstate(), "torch": torch.get_rng_state(),
            "numpy": [state[0], state[1].tolist(), int(state[2]), int(state[3]), float(state[4])],
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def restore_rng(state):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"])
    n = state["numpy"]
    np.random.set_state((n[0], np.asarray(n[1], dtype=np.uint32), n[2], n[3], n[4]))
    if torch.cuda.is_available() and state["cuda"]:
        torch.cuda.set_rng_state_all(state["cuda"])


def run_config(args, digest):
    names = ("architecture", "image_size", "cache_size", "batch_size", "workers", "seed", "epochs", "warmup_epochs",
             "patience", "mixup", "lr", "head_lr", "weight_decay", "threshold")
    return {**{name: getattr(args, name) for name in names}, "split_digest": digest,
            "preprocess": PREPROCESS, "recipe": "colab-v1"}


def checkpoint_metadata(args, digest):
    return {"classes": CLASSES, "preprocess": PREPROCESS, "architecture": args.architecture,
            "image_size": args.image_size, "cache_size": args.cache_size,
            "split_digest": digest, "temperature": 1., "threshold": args.threshold}


def validate_checkpoint(ck, args, digest):
    for key, value in checkpoint_metadata(args, digest).items():
        if key not in ("temperature", "threshold") and ck.get(key) != value:
            raise ValueError(f"Checkpoint contract mismatch: {key}")

## 8 · Eğitim döngüsü
İki epoch sınıflandırıcı ısınması, ardından ince ayar. Sınıf ağırlıklı kayıp, isteğe bağlı MixUp,
karma hassasiyet ve doğrulama QWK'sına göre erken durdurma kullanılır.
Bu hücre fonksiyonu tanımlar; tam eğitim 10. adımda başlar.

In [ ]:
def train(args):
    rows, digest = checked_data(args)
    smoke = args.action == "smoke"
    device = device_for(args)
    args.output.mkdir(parents=True, exist_ok=True)
    last, best_path = args.output / "last.pt", args.output / "best.pt"
    if (last.exists() or best_path.exists()) and not args.resume:
        raise FileExistsError("Run exists. Choose another output directory or use --resume.")
    if args.resume and not (last.exists() or best_path.exists()):
        raise FileNotFoundError("No last.pt or best.pt to resume")
    if args.resume and smoke:
        raise ValueError("Smoke checks do not resume")
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.set_num_threads(4)
    train_loader, val_loader = loader(rows["train"], args, True, smoke), loader(rows["val"], args, False, smoke)
    config = run_config(args, digest)
    model = build_model(pretrained=not (args.resume or smoke), architecture=args.architecture).to(device)
    counts = np.bincount([int(r["diagnosis"]) for r in rows["train"]], minlength=5)
    if np.any(counts == 0):
        raise ValueError("Every training class must be represented")
    weights = torch.tensor(1 / np.sqrt(counts), dtype=torch.float32, device=device)
    weights /= weights.mean()
    # One balancing method: class-weighted loss, no oversampling.
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW([{"params": model.features.parameters(), "lr": args.lr},
                                   {"params": model.classifier.parameters(), "lr": args.head_lr}],
                                  weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=.5, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    start, best, stale = 0, -float("inf"), 0
    if args.resume:
        checkpoints = [torch.load(path, map_location="cpu", weights_only=True)
                       for path in (last, best_path) if path.exists()]
        if any(checkpoint["config"] != config for checkpoint in checkpoints):
            raise ValueError("Resume config differs. Preserve model, split, cache and training parameters.")
        # best is saved first. A crash before last is replaced can leave best one
        # epoch ahead (or the only file): resume its complete optimizer/RNG state.
        ck = max(checkpoints, key=lambda checkpoint: checkpoint["epoch"])
        if not best_path.exists():
            raise FileNotFoundError("Best checkpoint missing; restore it before resuming this run")
        model.load_state_dict(ck["state_dict"])
        optimizer.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])
        scheduler.load_state_dict(ck["scheduler"])
        start, best, stale = ck["epoch"] + 1, ck["best_qwk"], ck["stale"]
        restore_rng(ck["rng"])
    atomic_json({**config, "torch": str(torch.__version__), "device": str(device),
                 "initialization": "random (smoke only)" if smoke else "ImageNet1K",
                 "splits": {s: len(r) for s, r in rows.items()}}, args.output / "config.json")
    for epoch in range(start, 1 if smoke else args.epochs):
        if not smoke and stale >= args.patience and epoch >= args.warmup_epochs + 3:
            break
        started, warmup = time.time(), epoch < args.warmup_epochs and not smoke
        for parameter in model.features.parameters():
            parameter.requires_grad_(not warmup)
        model.train()
        for module in model.features.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
        total, n = 0., 0
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            use_mixup = args.mixup > 0 and not warmup and len(y) > 1
            lam = float(np.random.beta(args.mixup, args.mixup)) if use_mixup else 1.
            order = torch.randperm(len(y), device=device)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                output = model(lam * x + (1 - lam) * x[order] if use_mixup else x)
                loss = lam * criterion(output, y) + (1 - lam) * criterion(output, y[order])
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            scaler.step(optimizer)
            scaler.update()
            total += float(loss.detach()) * len(y)
            n += len(y)
            if step % 40 == 0:
                print(f"Epoch {epoch + 1}/{args.epochs}: batch {step + 1}/{len(train_loader)}, loss={float(loss.detach()):.4f}", flush=True)
        labels, logits = evaluate(model, val_loader, device)
        if logits.shape != (len(labels), 5) or not torch.isfinite(logits).all():
            raise RuntimeError("Invalid validation output")
        m = metrics(labels.numpy(), logits.numpy())
        if smoke:
            atomic_save({**checkpoint_metadata(args, digest), "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                         "display_name": "SMOKE ONLY - untrained", "smoke_only": True}, args.output / "smoke_only.pt")
            atomic_json({"status": "passed", "batches": len(train_loader), "validation_examples": len(labels),
                         "architecture": args.architecture, "device": str(device)}, args.output / "smoke.json")
            print("SMOKE PASSED: real image forward/backward + validation + portable checkpoint. Not a trained model.", flush=True)
            return
        if m["qwk"] is None:
            raise RuntimeError("Validation QWK is undefined")
        improved = m["qwk"] > best
        best, stale = (m["qwk"], 0) if improved else (best, stale + 1)
        scheduler.step(m["qwk"])
        state = {**checkpoint_metadata(args, digest), "state_dict": model.state_dict(), "epoch": epoch,
                 "best_qwk": best, "stale": stale, "config": config, "rng": rng_state(),
                 "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict()}
        if improved:
            atomic_save(state, best_path)
        atomic_save(state, last)
        event = {"epoch": epoch + 1, "loss": total / n, "seconds": time.time() - started, **m}
        with (args.output / "history.jsonl").open("a", encoding="utf-8") as f:
            f.write(json.dumps(event) + "\n")
        atomic_json({"status": "training", "last_epoch": epoch + 1, "best_val_qwk": best}, args.output / "status.json")
        print(f"Validation QWK={m['qwk']:.4f}, macro F1={m['macro_f1']:.4f}, severe recall={m['classes']['Severe']['recall']:.4f}; best={best:.4f}", flush=True)
    atomic_json({"status": "trained", "next": "export; test only after final model selection", "best_val_qwk": best}, args.output / "status.json")
    print("TRAINING COMPLETE. No test images evaluated. Next: export.", flush=True)

## 9 · Kısa çalışma kontrolü
İki küçük eğitim batch'i ve doğrulama çalıştırılır. Rastgele başlangıçlı bu kontrol başarı ölçümü değildir.
Bellek hatası olursa 3. hücrede batch boyutunu düşürüp 3. ve 9. hücreleri yeniden çalıştırın.

In [ ]:
from datetime import datetime
smoke_args = copy(args)
smoke_args.action = "smoke"
smoke_args.output = Path("/content") / ("aptos_smoke_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
smoke_args.resume = False
train(smoke_args)
torch.cuda.empty_cache()

## 10 · Eğitimi başlat / kaldığı yerden devam et
Tam eğitim ImageNet ağırlıklarıyla başlar. `last.pt` veya `best.pt` varsa en yeni tam kayıttan devam eder.
Oturum kesilirse aynı `RUN_NAME` ve ayarlarla önceki hücreleri tekrar çalıştırın.

In [ ]:
training_args = copy(args)
training_args.action = "train"
training_args.resume = any((RUN / name).exists() for name in ("last.pt", "best.pt"))
train(training_args)
torch.cuda.empty_cache()

## 11 · Doğrulama, kalibrasyon ve masaüstü modeli
Kalibrasyon yalnızca doğrulama kümesini kullanır. Model seçimini QWK, macro F1 ve ağır evre duyarlılığıyla yapın.
En iyi epoch masaüstünün tanıdığı `.pt` dosyasına aktarılır.

In [ ]:
def export(args):
    rows, digest = checked_data(args)
    device = device_for(args)
    ck = torch.load(args.output / "best.pt", map_location="cpu", weights_only=True)
    validate_checkpoint(ck, args, digest)
    if args.threshold != ck["threshold"]:
        raise ValueError("Export threshold must match this run's preregistered threshold")
    if (args.output / "test_metrics.json").exists():
        raise RuntimeError("Final test recorded; do not recalibrate this run after seeing test results.")
    model = build_model(architecture=args.architecture).to(device)
    model.load_state_dict(ck["state_dict"])
    labels, logits = evaluate(model, loader(rows["val"], args), device)
    candidates = torch.logspace(math.log10(.25), math.log10(8.), 161)
    temperature = float(min(candidates, key=lambda t: float(nn.functional.cross_entropy(logits / t, labels))))
    report = {**metrics(labels.numpy(), logits.numpy()), **selective_metrics(labels, logits, temperature, args.threshold),
              "nll_before": float(nn.functional.cross_entropy(logits, labels)),
              "nll_after": float(nn.functional.cross_entropy(logits / temperature, labels)),
              "selected_epoch": ck["epoch"] + 1, "split_digest": digest}
    destination = args.output / f"aptos_{args.architecture}.pt"
    artifact = {**checkpoint_metadata(args, digest), "state_dict": ck["state_dict"], "temperature": temperature,
                "selected_epoch": ck["epoch"] + 1, "initialization": "ImageNet1K",
                "display_name": args.architecture.replace("_", " ") + " / APTOS Colab",
                "validation_metrics": report, "training_recipe": "colab-v1"}
    atomic_save(artifact, destination)
    atomic_json(report, args.output / "validation_metrics.json")
    atomic_json({"status": "exported", "model": destination.name, "test_evaluated": False}, args.output / "status.json")
    print(f"EXPORTED: {destination}. Validation calibration only; no test evaluation.", flush=True)

export_args = copy(args)
export_args.action = "export"
export(export_args)
validation = json.loads((RUN / "validation_metrics.json").read_text())
print("Doğrulama QWK:", round(validation["qwk"], 4))
print("Macro F1:", round(validation["macro_f1"], 4))
print("Ağır evre duyarlılığı:", round(validation["classes"]["Severe"]["recall"], 4))
print("Kabul oranı:", round(validation["coverage"], 4))

import matplotlib.pyplot as plt
history_path = RUN / "history.jsonl"
history = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()] if history_path.exists() else []
by_epoch = {r["epoch"]: r for r in history}
ordered = [by_epoch[k] for k in sorted(by_epoch)]
if ordered:
    plt.figure(figsize=(9, 3))
    for key, title in [("qwk", "QWK"), ("macro_f1", "Macro F1")]:
        plt.plot([r["epoch"] for r in ordered], [r[key] for r in ordered], label=title)
    plt.xlabel("Epoch")
    plt.ylabel("Doğrulama skoru")
    plt.legend()
    plt.grid(alpha=.2)
    plt.show()

## 12 · Son değerlendirme — son model seçildikten sonra
Varsayılan olarak atlanır. Model ve ayar kararını doğrulama kümesinde verdikten sonra True yapın.
Bu test kümesi B0 geliştirmesinde daha önce incelendi; sonuç yeni dış/klinik doğrulama değildir.

In [ ]:
def test(args):
    if not args.confirm_final_model:
        raise RuntimeError("Select final model on validation; then pass --confirm-final-model explicitly.")
    if (args.output / "test_metrics.json").exists():
        raise FileExistsError("Final test recorded. Read test_metrics.json; do not retune using this set.")
    rows, digest = checked_data(args)
    device = device_for(args)
    checkpoint = args.output / f"aptos_{args.architecture}.pt"
    ck = torch.load(checkpoint, map_location="cpu", weights_only=True)
    validate_checkpoint(ck, args, digest)
    model = build_model(architecture=args.architecture).to(device)
    model.load_state_dict(ck["state_dict"])
    labels, logits = evaluate(model, loader(rows["test"], args), device)
    report = {**metrics(labels.numpy(), logits.numpy()),
              **selective_metrics(labels, logits, ck["temperature"], ck["threshold"]),
              "split_digest": digest, "model_sha256": hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
              "note": "Held-out labeled APTOS train subset; competition test.csv has no public labels. This set was inspected for the B0 baseline, so it is a development benchmark, not fresh external validation."}
    atomic_json(report, args.output / "test_metrics.json")
    scores, predicted = (logits / ck["temperature"]).softmax(1).max(1)
    with (args.output / "test_predictions.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id_code", "true", "predicted", "score", "accepted"])
        for row, y, p, score in zip(rows["test"], labels, predicted, scores):
            writer.writerow([row["id_code"], int(y), int(p), float(score), float(score) >= ck["threshold"]])
    atomic_json({"status": "final_test_complete", "model": checkpoint.name}, args.output / "status.json")
    print(json.dumps(report, indent=2), flush=True)

FINAL_MODEL_SELECTED = False
if FINAL_MODEL_SELECTED:
    test_args = copy(args)
    test_args.action = "test"
    test_args.confirm_final_model = True
    test(test_args)
else:
    print("Son test atlandı; model seçimi doğrulama kümesinde yapılır.")

## 13 · Modeli indir
İndirdiğiniz `.pt` dosyasını masaüstü uygulamasındaki **Model ekle…** düğmesiyle seçin.
Kayıtlar ve sonuçlar Drive'daki deneme klasöründe kalır.

In [ ]:
from google.colab import files
MODEL = RUN / f"aptos_{ARCHITECTURE}.pt"
if not MODEL.is_file():
    raise FileNotFoundError("Önce eğitim ve dışa aktarma adımlarını tamamlayın.")
files.download(str(MODEL))

Birebir kopya ayıklama, hasta bazında ayrımı veya yakın kopya ayıklamayı garanti etmez.
Skor eşiği klinik güvence değildir. Yeni modellerin başarısı tam eğitim ve değerlendirmeden sonra ölçülür.

Kaynak kod mevcut Blindless Detection projesinden türetilmiştir.
MIT License

Copyright (c) 2025 Olgun Yılmaz

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.